# Sensor Detection & Cataloging

OceanStream automatically detects sensors from data columns and provides metadata from a built-in sensor catalog.

In [ ]:
# Setup
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

## Sensor Catalogue

In [ ]:
from oceanstream import get_sensor_catalogue

catalogue = get_sensor_catalogue()

print(f"📋 Sensor Catalogue: {len(catalogue.sensors)} sensors registered")

In [ ]:
# List all available sensors
print("Available sensors:")
for sensor_id in sorted(catalogue.sensors.keys()):
    sensor = catalogue.sensors[sensor_id]
    print(f"   • {sensor_id}: {sensor.name}")

## Sensor Details

In [ ]:
# Get details for a specific sensor
sensor = catalogue.get_sensor("airmar-150wx")

if sensor:
    print(f"🔬 Sensor: {sensor.name}")
    print(f"   ID: {sensor.id}")
    print(f"   Manufacturer: {sensor.manufacturer}")
    print(f"   Model: {sensor.model}")
    print(f"   Variables: {sensor.variables[:5]}...")  # First 5
    print(f"   Description: {sensor.description[:100]}...")

## Auto-Detection from Data

In [ ]:
import pandas as pd

# Sample data columns (typical Saildrone data)
sample_columns = [
    'time', 'latitude', 'longitude',
    'TEMP_AIR_MEAN', 'RH_MEAN', 'BARO_PRES_MEAN',  # Weather
    'TEMP_SBE37_MEAN', 'SAL_SBE37_MEAN',           # CTD
    'CHLOR_WETLABS_MEAN', 'CDOM_WETLABS_MEAN',     # Fluorometer
    'UWND_MEAN', 'VWND_MEAN',                       # Wind
]

# Detect sensors from columns
detected = catalogue.detect_sensors(sample_columns)
print(f"🔍 Detected {len(detected)} sensors from columns:")
for sensor_id in detected:
    sensor = catalogue.get_sensor(sensor_id)
    print(f"   • {sensor.name} ({sensor_id})")

## Sensor Definitions Location

In [ ]:
# Sensor definitions are JSON files
definitions_dir = project_root / "oceanstream" / "sensors" / "definitions"

if definitions_dir.exists():
    sensor_dirs = [d.name for d in definitions_dir.iterdir() if d.is_dir()]
    print(f"📁 Sensor definitions in: {definitions_dir}")
    print(f"   {len(sensor_dirs)} sensors defined")
    for s in sorted(sensor_dirs)[:5]:
        print(f"   • {s}/sensor.json")

## STAC Instrument Format

Detected sensors are included in STAC metadata as instruments:

```json
{
  "summaries": {
    "instruments": [
      "airmar-150wx (Airmar WeatherStation 150WX)",
      "sbe37-odo (Sea-Bird SBE37 with DO)",
      "wetlabs-eco-flntu (WET Labs ECO FLNTU)"
    ]
  }
}
```

## Adding Custom Sensors

Create a new sensor definition at `oceanstream/sensors/definitions/<sensor-id>/sensor.json`:

```json
{
  "id": "my-sensor",
  "name": "My Custom Sensor",
  "manufacturer": "ACME",
  "model": "Model X",
  "description": "Description of the sensor",
  "variables": ["var1", "var2", "var3"],
  "detection_patterns": ["VAR1_", "VAR2_"]
}
```